# TrafficVision: CNN Traffic Sign Classifier

You are building a traffic sign recognition system. The dataset contains **32x32 RGB images** of **10 types** of traffic signs. Your goal is to design, build, and train a CNN from scratch that classifies these signs.

---

## Dataset

The file `traffic_signs.npz` contains:

| Key | Shape | Description |
|-----|-------|-------------|
| `X_train` | (1000, 32, 32, 3) | Training images (uint8, 0-255) |
| `y_train` | (1000,) | Training labels (0-9) |
| `X_val` | (300, 32, 32, 3) | Validation images |
| `y_val` | (300,) | Validation labels |

---

## Tasks

1. **Prepare the data**: Create a PyTorch `Dataset` and `DataLoader` objects for training and validation. Scale pixel values to the 0-1 range and rearrange images to channels-first format, since your model will be evaluated on data prepared this way

2. **Design and build a CNN** as a subclass of `nn.Module`. The architecture is up to you, but it must contain at least 2 convolutional layers and at least 1 pooling layer, and end with fully connected layers producing 10 output units

3. **Train the model** using `CrossEntropyLoss` and an optimizer of your choice, and store the trained network in a variable named `model`

4. **Evaluate on the validation set**. Your model must achieve at least **70% accuracy** on a hidden test set of unseen traffic signs

---

## Expected Output

| Variable | Description |
|----------|-------------|
| `model` | Your trained CNN classifier (an `nn.Module` instance) |

---

## Evaluation Criteria

Your submission will be checked for:

- Code runs without errors
- A trained `nn.Module` exists in the variable `model`
- The model contains **at least 2 Conv2d layers**
- The model contains **at least 1 pooling layer**
- The model outputs **10 units** for a (1, 3, 32, 32) input
- The model achieves **>= 70% accuracy** on the hidden test set

In [1]:
# Run this cell before writing your solution
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Load dataset
data = np.load("traffic_signs.npz")
X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]

NUM_CLASSES = 10

print(f"Training set: {X_train.shape}, Labels: {y_train.shape}")
print(f"Validation set: {X_val.shape}, Labels: {y_val.shape}")

Training set: (1000, 32, 32, 3), Labels: (1000,)
Validation set: (300, 32, 32, 3), Labels: (300,)


1. Create PyTorch Datase and DataLoader

In [3]:
# Write your code here
class TrafficSignDataset(Dataset):
  def __init__(self, images, labels):
    self.images = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    self.labels = torch.tensor(labels, dtype=torch.long)
  
  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    return self.images[idx], self.labels[idx]

train_dataset = TrafficSignDataset(X_train, y_train)
val_dataset = TrafficSignDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

2. Design and implement a Convolutional Neural Network (CNN)

In [6]:
class TrafficCNN(nn.Module):
  def __init__(self):
    super(TrafficCNN, self).__init__()
    self.features = nn.Sequential(
      nn.Conv2d(3, 16, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(2, 2),
      nn.Conv2d(16, 32, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(2, 2)
    )
    self.classifier = nn.Sequential(
      nn.Flatten(),
      nn.Linear(32 * 8 * 8, 128),
      nn.ReLU(),
      nn.Linear(128, NUM_CLASSES)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x 

model = TrafficCNN()

3. Train the model using CrossEntropyLoss and Opitimizer

In [7]:
criterion =  nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15 
model.train()
for epoch in range(epochs):
  for images, labels in train_loader:
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

4. Evaluate the trained model on the validation set.

In [8]:
model.eval()
correct = 0
total = 0 
with torch.no_grad():
  for images, labels in val_loader:
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

val_accuracy = correct / total
print(f'Validation Accuracy: {val_accuracy:.4f}')

Validation Accuracy: 0.9900
